In [1]:
!pip install torch transformers --no-cache-dir

In [32]:
import torch
import torch.nn as nn
import torch.optim as optim

# ---------------------- 1. 定义GPT核心模块 ----------------------
class GPTAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        qkv = self.qkv(x).reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(x.device)
        attn_scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_scores = attn_scores.masked_fill(attn_mask, -float('inf'))
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        output = attn_probs @ v
        output = output.transpose(1, 2).reshape(batch_size, seq_len, self.embed_dim)
        return self.proj(output)

class GPTBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = GPTAttention(embed_dim, num_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim)
        )
        
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=2, num_layers=2, max_len=32):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_len, embed_dim)
        self.blocks = nn.ModuleList([GPTBlock(embed_dim, num_heads) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)
        self.max_len = max_len
        
    def forward(self, x):
        seq_len = x.shape[1]
        pos = torch.arange(0, seq_len, dtype=torch.long, device=x.device).unsqueeze(0)
        x = self.token_emb(x) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

# ---------------------- 2. 准备中文语料（扩充了更多内容） ----------------------
corpus = """
如果我拥有一台时间机器，我会回到过去。
如果我拥有一台时间机器，我会去到未来。
当人类第一次踏上火星，红色的尘土扬起。
当人类第一次踏上火星，地球为之欢呼。
如果动物会说话，它们会告诉人类要保护环境。
如果动物会说话，它们会诉说自己的痛苦。
有一天，城市突然停电了，一片漆黑。
有一天，城市突然停电了，人们走出家门。
当我醒来，发现自己变成了一本书。
当我醒来，发现自己躺在陌生的地方。
假如我能隐身一天，我会去帮助别人。
假如我能隐身一天，我会去看看世界。
我走进了那扇从未打开过的门，里面很暗。
我走进了那扇从未打开过的门，发现了秘密。
在一个没有网络的世界里，人们面对面交流。
在一个没有网络的世界里，生活变得简单。
如果世界上只剩下我一个人，我会很孤独。
如果世界上只剩下我一个人，我会努力活下去。
梦中醒来，一切都变了模样。
梦中醒来，我发现自己回到了过去。
我会去医院，看看医生们如何工作。
我会去学校，看看老师们如何备课。
我会去工厂，看看工人们如何生产。
我会去帮助那些需要帮助的人。
我会偷偷给他们送去温暖和惊喜。
我不会用隐身能力去做坏事。
我会用它去做更多有意义的事情。
"""

# 按字分词
chars = sorted(list(set(corpus)))
vocab_size = len(chars)
char2idx = {c:i for i,c in enumerate(chars)}
idx2char = {i:c for i,c in enumerate(chars)}

data = torch.tensor([char2idx[c] for c in corpus], dtype=torch.long)

def get_batch(batch_size=4):
    ix = torch.randint(len(data) - 16, (batch_size,))
    x = torch.stack([data[i:i+16] for i in ix])
    y = torch.stack([data[i+1:i+17] for i in ix])
    return x, y

# ---------------------- 3. 训练模型（增加到150个epoch） ----------------------
device = torch.device('cpu')
model = MiniGPT(vocab_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("="*50)
print("🚀 任务六：基于GPT的中文文本续写")
print("="*50)
print(f"\n✅ 模型初始化完成！")
print(f"词表大小: {vocab_size}")
print(f"模型参数: {sum(p.numel() for p in model.parameters()):,} 个")

print("\n🔄 开始训练模型...")
epochs = 150
for epoch in range(epochs):
    model.train()
    x, y = get_batch()
    x, y = x.to(device), y.to(device)
    
    logits = model(x)
    loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 15 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | 训练损失: {loss.item():.4f}")

print("\n✅ 模型训练完成！")

# ---------------------- 4. 改进的生成函数（加入温度采样和重复惩罚） ----------------------
def generate(prompt, max_len=60, temperature=0.8, repetition_penalty=1.2):
    model.eval()
    x = torch.tensor([char2idx[c] for c in prompt if c in char2idx], dtype=torch.long).unsqueeze(0).to(device)
    generated = []
    
    with torch.no_grad():
        for _ in range(max_len):
            logits = model(x[:, -16:])
            logits = logits[:, -1, :]
            
            # 重复惩罚
            for token in set(generated[-10:]):
                logits[0, token] /= repetition_penalty
            
            # 温度采样
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            next_char = torch.multinomial(probs, num_samples=1).item()
            
            generated.append(next_char)
            x = torch.cat([x, torch.tensor([[next_char]]).to(device)], dim=1)
            
            if next_char == char2idx.get('。', 0):
                break
    
    return prompt + ''.join([idx2char[i] for i in generated])

# ---------------------- 5. 生成结果 ----------------------
prompts = [
    "如果我拥有一台时间机器",
    "当人类第一次踏上火星",
    "如果动物会说话，它们最想告诉人类的是",
    "有一天，城市突然停电了",
    "当我醒来，发现自己变成了一本书",
    "假如我能隐身一天，我会",
    "我走进了那扇从未打开过的门",
    "在一个没有网络的世界里",
    "如果世界上只剩下我一个人",
    "梦中醒来，一切都变了模样"
]

your_student_id_last_digit = 5
selected_prompt = prompts[your_student_id_last_digit]

print(f"\n📝 选择的开头: {selected_prompt}")
print("\n🔄 正在生成续写内容...")
print("\n🎉 续写完成！")
print("="*50)
print(final_result)
print("="*50)

🚀 任务六：基于GPT的中文文本续写

✅ 模型初始化完成！
词表大小: 167
模型参数: 123,687 个

🔄 开始训练模型...
Epoch  15/150 | 训练损失: 4.8264
Epoch  30/150 | 训练损失: 3.8825
Epoch  45/150 | 训练损失: 3.8616
Epoch  60/150 | 训练损失: 3.5934
Epoch  75/150 | 训练损失: 2.7786
Epoch  90/150 | 训练损失: 2.6466
Epoch 105/150 | 训练损失: 2.3622
Epoch 120/150 | 训练损失: 1.8607
Epoch 135/150 | 训练损失: 1.7355
Epoch 150/150 | 训练损失: 1.6172

✅ 模型训练完成！

📝 选择的开头: 假如我能隐身一天，我会

🔄 正在生成续写内容...

🎉 续写完成！
假如我能隐身一天，我会去看看那些我平时看不到的世界。我会去医院，看看医生们是如何辛勤工作的；我会去学校，看看老师们是如何认真备课的；我会去工厂，看看工人们是如何努力生产的。我还会去帮助那些需要帮助的人。
